In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *

In [0]:
schema = StructType([
    StructField("event_id", StringType(), False),
    StructField("expense_id", IntegerType(), False),
    StructField("employee_id", IntegerType(), True),
    StructField("region_id", IntegerType(), True),
    StructField("expense_type", StringType(), True),
    StructField("expense_amount", LongType(), True),
    StructField("event_time", TimestampType(), False),
    StructField("ingestion_time", TimestampType(), False)
])

exp_df = spark.readStream.format("delta").table("sales_project_streaming.brz.expenses")

parsed_df = (exp_df.withColumn("parsed", F.from_json("value", schema))
             .select("parsed"))

exp_parsed = (parsed_df.select("parsed.expense_id", "parsed.employee_id",
                              "parsed.region_id", "parsed.expense_type",
                              "parsed.expense_amount", "parsed.event_time")
              .withColumn("hour", F.hour("event_time"))
              .withColumn("event_date", F.to_date("event_time"))
              .withColumn("day", F.date_format("event_date", "EEEE"))
              .withColumn("day_of_week", F.dayofweek("event_date"))
              .withColumn("month", F.date_format(F.col("event_date"), "MMMM"))
              .withColumn("is_weekend", F.col("day_of_week").isin([1,7]))
              .withColumn("week_of_month", F.weekofyear("event_date") -
                                            F.weekofyear(
                                                F.date_sub(F.col("event_date"), F.day(F.col("event_date")+1))
                                            )+1)
              .withColumn("year", F.year(F.col("event_date")))
              .withColumn("processed_time", F.current_timestamp())
                                            )

good_exp = (exp_parsed.filter((F.col("employee_id").isNotNull()) & (F.col("region_id").isNotNull())
                              & (F.col("expense_type").isNotNull()) & (F.col("expense_amount")>=0))
            .withColumn("expense_type", F.initcap("expense_type")))

bad_exp = (exp_parsed.filter((F.col("employee_id").isNull()) | (F.col("region_id").isNull())
                              | (F.col("expense_type").isNull()) | (F.col("expense_amount")<0))
            )

good_query = (good_exp.writeStream
              .format("delta")
              .option("checkpointLocation", "/Volumes/sales_project_streaming/slv/checkpoints_vol/exp_slv_chck/")
              .outputMode("append")
              .trigger(availableNow = True)
              .table("sales_project_streaming.slv.expenses"))

bad_query = (bad_exp.writeStream
              .format("delta")
              .option("checkpointLocation", "/Volumes/sales_project_streaming/slv/checkpoints_vol/bad_exp_chck_slv/")
              .outputMode("append")
              .trigger(availableNow = True)
              .table("sales_project_streaming.brz.bad_records_expenses"))

good_query.awaitTermination()
bad_query.awaitTermination()

In [0]:
%sql
select * from sales_project_streaming.slv.expenses;

In [0]:
# %sql

# update sales_project_streaming.slv.expenses
# set year = year(event_date);

# select * from sales_project_streaming.slv.expenses;

In [0]:
# %sql
# alter table sales_project_streaming.slv.sales
# add columns year bigint after week_of_month;

In [0]:
# %sql
# select * from sales_project_streaming.brz.bad_records_expenses;

In [0]:
display(spark.sql("select * from sales_project_streaming.brz.regions"))

schema = StructType([
    StructField("event_id", StringType(), False),
    StructField("region_id", IntegerType(), False),
    StructField("region_name", StringType(), False),
    StructField("operation", StringType(), False),
    StructField("event_time", TimestampType(), False),
    StructField("ingestion_time", TimestampType(), False)
])

reg_raw = (spark.readStream.format("delta")
           .table("sales_project_streaming.brz.regions"))

reg_parsed = (reg_raw.withColumn("parsed", F.from_json("value", schema))
              .select("parsed"))

reg_df = reg_parsed.select("parsed.region_id", "parsed.region_name")

query = (reg_df.writeStream
         .format("delta")
         .option("checkpointLocation", "/Volumes/sales_project_streaming/slv/checkpoints_vol/regions_chck_Slv/")
         .outputMode("append")
         .trigger(availableNow = True)
         .table("sales_project_streaming.slv.regions"))

query.awaitTermination()

In [0]:
# %sql
# select * from sales_project_streaming.slv.regions